# Day 1 practice — Reading HTTP for real

**Read first:** [01_theory_http_and_rest.md](01_theory_http_and_rest.md)

By the end of this notebook you will have sent real requests, read every part of a response,
produced a `404` and a `422` on purpose, proved to yourself why `POST` is dangerous to retry,
and redesigned a badly-shaped API.

**How to use this notebook:** every 🔮 cell asks you to predict before you run the next one.
Guessing first is not a formality — it is the thing that makes today stick. Write your guess in the
cell, then run.

## Setup

Nothing here needs a database. You do need the module environment installed:

```bash
cd module-03-fastapi-azure
python -m venv .venv
source .venv/bin/activate      # Windows: .venv\Scripts\activate
pip install -r requirements.txt
```

In [ ]:
import json
from datetime import date, datetime
from decimal import Decimal

import requests

print("ready")

---

## Part 1 — One real request, taken apart

We'll call Open-Meteo, the same free API your Module 2 pipeline uses. No key, no signup.

In [ ]:
response = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params={                       # requests builds ?latitude=52.09&longitude=5.12&... for you
        "latitude": 52.09,         # Utrecht
        "longitude": 5.12,
        "current": "temperature_2m",
    },
    timeout=30,                    # ALWAYS. A hung request should fail, not wait forever.
)

print("status code :", response.status_code)
print("final URL   :", response.url)      # note how params were encoded

### 🔮 Predict

Before running the next cell: what do you expect `response.headers` to contain? Name two headers you
are fairly confident will be there, and what each is for.

*(Your guess: ...)*

In [ ]:
for name, value in response.headers.items():
    print(f"{name:<28} {value}")

`Content-Type: application/json` is the server telling you how to interpret the bytes.
That is exactly what lets the next cell work.

In [ ]:
print("--- .text  (raw bytes, decoded to a str) ---")
print(type(response.text), "|", response.text[:120], "...")

print()
print("--- .json() (parsed into Python objects) ---")
data = response.json()
print(type(data))
print(json.dumps(data, indent=2)[:400])

In [ ]:
# The difference matters: one you can index, the other you cannot.
print("data['current']['temperature_2m'] =", data["current"]["temperature_2m"])

try:
    response.text["current"]           # a str indexed by a str
except TypeError as exc:
    print("response.text['current'] ->", type(exc).__name__, ":", exc)

### `raise_for_status()` — turning silence into noise

A `404` is still a *successful HTTP exchange*. `requests` will not raise on it. If you forget to
check, your code sails on with an error page where it expected data.

In [ ]:
bad = requests.get("https://api.open-meteo.com/v1/this-endpoint-does-not-exist", timeout=30)

print("status code       :", bad.status_code)
print("did requests raise? no - we got here fine")
print("body preview      :", bad.text[:120])

try:
    bad.raise_for_status()             # THIS is what makes it loud
except requests.HTTPError as exc:
    print()
    print("raise_for_status() ->", type(exc).__name__)
    print(exc)

> 🎯 **Remember this** — `requests` does not raise on `4xx`/`5xx`. `raise_for_status()` is the one
> line that turns a silent wrong answer into a crash you can see. Your Module 2 `fetch.py` calls it
> for exactly this reason.

---

## Part 2 — Status codes, on demand

Rather than hunting the internet for a server that returns a `422`, let's build a tiny API that
returns whatever we ask for. This is a preview of Day 2 — read it, don't worry about the syntax yet.

`TestClient` calls the app directly, in-process. No server, no port, no network.

In [ ]:
from fastapi import FastAPI, HTTPException, status
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

demo = FastAPI()

@demo.get("/ok")
def ok():
    return {"message": "here you go"}

@demo.post("/things", status_code=status.HTTP_201_CREATED)
def create_thing():
    return {"id": 1, "created": True}

@demo.delete("/things/1", status_code=status.HTTP_204_NO_CONTENT)
def delete_thing():
    return None

@demo.get("/things/{thing_id}")
def read_thing(thing_id: int):
    if thing_id != 1:
        raise HTTPException(status_code=404, detail=f"No thing with id {thing_id}")
    return {"id": 1}

@demo.get("/boom")
def boom():
    raise ValueError("something went wrong inside our code")

client = TestClient(demo, raise_server_exceptions=False)
print("demo API ready")

### 🔮 Predict

Fill in the status code you expect for each row, *then* run the cell.

| Request | Your guess |
|---|---|
| `GET /ok` | |
| `POST /things` | |
| `DELETE /things/1` | |
| `GET /things/999` | |
| `GET /things/abc` | |
| `GET /boom` | |

In [ ]:
calls = [
    ("GET",    "/ok"),
    ("POST",   "/things"),
    ("DELETE", "/things/1"),
    ("GET",    "/things/999"),
    ("GET",    "/things/abc"),
    ("GET",    "/boom"),
]

for method, path in calls:
    r = client.request(method, path)
    family = f"{r.status_code // 100}xx"
    whose = {"2": "worked", "4": "CALLER's fault", "5": "SERVER's fault"}.get(str(r.status_code)[0], "")
    print(f"{method:<7} {path:<15} -> {r.status_code}  ({family}, {whose})")
    if r.content:
        print(f"{'':<24}    body: {r.text[:90]}")

Three things to notice:

1. **`DELETE` returned `204` with an empty body.** "It worked, and there is deliberately nothing to
   tell you." Not `200` with `{"deleted": true}` — the status code already said that.
2. **`GET /things/abc` returned `422`, not `404`.** The path parameter is typed `int`, so `"abc"`
   never reached your function. The doorman rejected it. That's Day 2's topic and it happened here
   for free.
3. **`GET /boom` returned `500`.** Our code raised an exception we didn't handle. `5xx` = our fault.
   In production that's the one that pages somebody.

### 🔁 Recall check

Look at the `422` body below and find the field name inside it.

In [ ]:
r = client.get("/things/abc")
print(json.dumps(r.json(), indent=2))

`loc` is `["path", "thing_id"]` — a breadcrumb trail reading *outside in*: in the path, the field
`thing_id`. On Day 3 this becomes `["body", "location", "lat"]` for nested data, and it will save
you a great deal of guessing.

---

## Part 3 — Safe and idempotent, felt rather than read

The words are abstract until you watch them misbehave. Let's give the demo API a real store.

In [ ]:
orders: dict[int, dict] = {}
next_id = {"value": 1}

app2 = FastAPI()

class OrderIn(BaseModel):
    item: str
    quantity: int = Field(ge=1)

@app2.post("/orders", status_code=201)
def create_order(order: OrderIn):
    oid = next_id["value"]
    next_id["value"] += 1
    orders[oid] = {"id": oid, **order.model_dump()}
    return orders[oid]

@app2.put("/orders/{oid}")
def replace_order(oid: int, order: OrderIn):
    orders[oid] = {"id": oid, **order.model_dump()}
    return orders[oid]

@app2.delete("/orders/{oid}", status_code=204)
def delete_order(oid: int):
    orders.pop(oid, None)          # pop with a default: deleting twice is fine
    return None

@app2.get("/orders")
def list_orders():
    return list(orders.values())

c2 = TestClient(app2)
print("order API ready")

### 🔮 Predict

You are going to send the **same** `POST /orders` twice, then the **same** `PUT /orders/1` twice,
then the **same** `DELETE /orders/1` twice.

How many orders exist after each pair? Write your three numbers down first.

In [ ]:
def show(label):
    print(f"{label:<34} -> {len(c2.get('/orders').json())} order(s): {c2.get('/orders').json()}")

payload = {"item": "soup", "quantity": 1}

c2.post("/orders", json=payload)
show("after POST x1")
c2.post("/orders", json=payload)
show("after POST x2  (SAME body!)")

print()
c2.put("/orders/1", json={"item": "soup", "quantity": 5})
show("after PUT x1")
c2.put("/orders/1", json={"item": "soup", "quantity": 5})
show("after PUT x2")

print()
print("DELETE /orders/1 ->", c2.delete("/orders/1").status_code)
show("after DELETE x1")
print("DELETE /orders/1 ->", c2.delete("/orders/1").status_code)
show("after DELETE x2")

**`POST` twice gave you two orders. `PUT` twice and `DELETE` twice changed nothing the second
time.** That is idempotency, and now you have watched it rather than read it.

This is the same property as your Module 2 loader's `ON CONFLICT (city, obs_date) DO UPDATE` — the
**light switch**. Flipping it up when it's already up leaves it up.

It is also exactly why double-clicking "Pay now" is dangerous and double-clicking "Refresh" is not.

### 🔁 Recall check

<details>
<summary>The checkout page retries <code>POST /payments</code> after a network hiccup and the customer is charged twice. The method won't save you. What's the standard fix?</summary>

An **idempotency key**: the client generates a unique id once, sends it as a header on every attempt,
and the server records which keys it has already honoured and refuses to process one twice.

That is your Module 2 `ON CONFLICT` trick lifted up to the HTTP layer — a primary key on "requests I
have already handled".
</details>

---

## Part 4 — REST design: fix the bad menu

Below is a real-shaped bad API. Rewrite each line as `METHOD /path`.

In [ ]:
bad_api = [
    "POST /getUserById?id=42",
    "GET  /deleteCity?name=Utrecht",
    "POST /api/updateUserEmail",
    "GET  /listAllCitiesInCountry?country=NL",
    "POST /createOrderForUser?user=42",
]

for line in bad_api:
    print(line)

print()
print("Write your redesign below, then open the solution.")

<details>
<summary>💡 Solution</summary>

| Bad | Good | Why |
|---|---|---|
| `POST /getUserById?id=42` | `GET /users/42` | Reading is `GET`; the id identifies the resource, so it belongs in the path |
| `GET /deleteCity?name=Utrecht` | `DELETE /cities/Utrecht` | **A `GET` must never destroy data.** Crawlers and link previewers fire GETs unprompted |
| `POST /api/updateUserEmail` | `PATCH /users/42` with `{"email": "..."}` | Partial edit is `PATCH`; the new value goes in the body |
| `GET /listAllCitiesInCountry?country=NL` | `GET /cities?country=NL` | The resource is "cities"; country is a **filter**, so it stays in the query string |
| `POST /createOrderForUser?user=42` | `POST /users/42/orders` | Orders are a sub-collection of a user; `POST` to a collection creates a member |

Read the good column aloud. Each is a sentence: **verb, then noun.** Nobody had to tell you what
`DELETE /cities/Utrecht` does — that predictability *is* REST.
</details>

---

## Part 5 — JSON traps

Four rules cause nearly every JSON error you will meet.

In [ ]:
broken = [
    ('{"city": "Utrecht",}',      "trailing comma"),
    ("{'city': 'Utrecht'}",       "single quotes (that's a Python dict, not JSON)"),
    ('{city: "Utrecht"}',         "unquoted key"),
    ('{"city": "Utrecht" // hi}', "comments are not allowed"),
]

for text, label in broken:
    try:
        json.loads(text)
        print(f"OK       {text}")
    except json.JSONDecodeError as exc:
        print(f"REJECTED {text:<28} {label}")
        print(f"         {exc}")

### The one that will actually bite you

Day 1 said JSON has no idea what a `date` or a `Decimal` is — and your Module 2 `marts` tables are
full of `NUMERIC` columns, which arrive in Python as `Decimal`.

In [ ]:
row = {
    "city": "Utrecht",
    "obs_date": date(2026, 9, 1),      # a real date object
    "temp_max_c": Decimal("21.40"),    # what Postgres NUMERIC becomes in Python
}

try:
    json.dumps(row)
except TypeError as exc:
    print("json.dumps(row) ->", type(exc).__name__)
    print(exc)

**Who should fix this, and where?** You *could* convert by hand in every handler that touches a
`NUMERIC` column — and forget one, eventually.

On Day 3 you'll declare `temp_max_c: float` on a Pydantic response model and the conversion happens
at the boundary, once, for every endpoint. That's the answer, and now you've felt the problem it
solves.

Here's the manual version, so you know what you're being saved from:

In [ ]:
def fallback(value):
    if isinstance(value, Decimal):
        return float(value)
    if isinstance(value, (date, datetime)):
        return value.isoformat()
    raise TypeError(f"cannot serialise {type(value).__name__}")

print(json.dumps(row, default=fallback, indent=2))

---

## Exercises

### Exercise 1 — Read a response with no help (⭐)

Call Open-Meteo for **Rotterdam** (lat 51.92, lon 4.48), asking for `daily=temperature_2m_max` over
the last 3 days, and print: the status code, the `Content-Type` header, and the list of max
temperatures. Use `raise_for_status()`.

In [ ]:
# Your code here


<details>
<summary>💡 Solution</summary>

```python
from datetime import timedelta

end = date.today() - timedelta(days=1)      # yesterday: today's archive is incomplete
start = end - timedelta(days=2)

r = requests.get(
    "https://archive-api.open-meteo.com/v1/archive",
    params={
        "latitude": 51.92,
        "longitude": 4.48,
        "start_date": start.isoformat(),
        "end_date": end.isoformat(),
        "daily": "temperature_2m_max",
        "timezone": "Europe/Amsterdam",
    },
    timeout=30,
)
r.raise_for_status()

print("status      :", r.status_code)
print("content-type:", r.headers["Content-Type"])
print("dates       :", r.json()["daily"]["time"])
print("max temps   :", r.json()["daily"]["temperature_2m_max"])
```

Note the **archive** host and the "ending yesterday" window — both copied straight from your Module 2
`fetch.py`, for the same reasons.
</details>

### Exercise 2 — Design the endpoints (⭐⭐)

You are designing an API for the Module 2 warehouse. Write `METHOD /path` for each need:

1. List every city that has data
2. Get the daily observations for Utrecht in August 2026, 20 per page
3. Get one weekly summary row for Utrecht for the week starting 2026-08-31
4. Trigger a fresh pipeline run
5. Remove all data for a city that was added by mistake

In [ ]:
# Your answers here (as comments or strings)


<details>
<summary>💡 Solution</summary>

```
1. GET    /cities
2. GET    /weather/daily?city=Utrecht&from=2026-08-01&to=2026-08-31&limit=20&offset=0
3. GET    /weather/weekly?city=Utrecht&week_start=2026-08-31
4. POST   /runs                      <- creating a "run" resource. NOT GET /triggerPipeline
5. DELETE /cities/Utrecht
```

Number 4 is the interesting one. "Do a thing" doesn't fit nouns-and-verbs neatly, and the standard
move is to **make the action a resource**: a pipeline run is a thing that gets created, has an id,
and can be inspected later with `GET /runs/17`. That is much more useful than a fire-and-forget
`POST /triggerPipeline`, because now the caller can poll for the result.

Number 3 could also be `GET /weather/weekly/Utrecht/2026-08-31`. Both are defensible; the query-string
version composes better with filtering, which is why the project uses it.
</details>

### Exercise 3 — Choose the status code (⭐⭐⭐)

For each situation, give the code you'd return **and one sentence of justification**.

1. A `POST /cities` succeeds and creates Zwolle
2. `GET /cities/Atlantis`, which does not exist
3. `POST /cities` where `latitude` is `"north-ish"`
4. `GET /weather/daily` while the database is down
5. `DELETE /cities/Zwolle` succeeds and there's nothing to return
6. A caller with a valid API key requests an admin-only endpoint
7. A caller with **no** API key requests that endpoint
8. Your code raises `ZeroDivisionError` computing an average

In [ ]:
# Your answers here


<details>
<summary>💡 Solution</summary>

| # | Code | Why |
|---|---|---|
| 1 | `201 Created` | A `POST` that created a resource. `200` would work but says less |
| 2 | `404 Not Found` | The resource doesn't exist. Caller's mistake, so `4xx` |
| 3 | `422 Unprocessable Entity` | The JSON parsed fine; a **value** failed validation. FastAPI does this for free |
| 4 | `503 Service Unavailable` | **We** are fine, a dependency is not. Not `500` — nothing in our code is broken |
| 5 | `204 No Content` | Success with deliberately nothing to send. Don't return `200 {"deleted": true}` |
| 6 | `403 Forbidden` | We know exactly who you are, and no |
| 7 | `401 Unauthorized` | Misnamed: it means *unauthenticated*. "Who are you?" |
| 8 | `500 Internal Server Error` | An unhandled exception in our code. Our fault. This is the one that pages somebody |

The 4 vs 8 distinction is the one worth internalising: **`503` means "try again later", `500` means
"a human needs to fix a bug".** Getting these backwards means either being woken up for other
people's typos, or *not* being woken up when your service is genuinely broken.
</details>

---

## ✅ Before you move on

Say each of these out loud without scrolling up:

- The four parts of a request, and the three parts of a response
- What `safe` and `idempotent` mean, and which methods are which
- What the first digit of a status code tells you
- Where a filter belongs: path or query string?
- Two things JSON cannot represent

Then open **[Day 2 theory](../day2-fastapi-basics/03_theory_fastapi_fundamentals.md)** and build the
counter you have been standing at all day.